**Handling Late Arriving Data Using PySpark and Delta Lake MERGE**

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

**STEP 1: CREATE EXISTING PROCESSED DATA**

In [0]:
target_data = [
    (1, "2024-01-01", 100),
    (2, "2024-01-02", 200)
]
target_df = spark.createDataFrame(
    target_data,
    ["id", "event_date", "amount"]
)
target_df = target_df.withColumn(
    "event_date",
    F.to_date("event_date")
)
table_name = "default.late_arriving_demo"

# CREATE DELTA TARGET TABLE
target_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

print("Existing Processed Data:")
display(spark.table(table_name))

Existing Processed Data:


id,event_date,amount
1,2024-01-01,100
2,2024-01-02,200


**STEP 2: CREATE INCOMING SOURCE DATA**

In [0]:
source_data = [
    (3, "2024-01-03", 300),  # New record
    (1, "2024-01-01", 150)   # Late-arriving update
]
source_df = spark.createDataFrame(
    source_data,
    ["id", "event_date", "amount"]
)
source_df = source_df.withColumn(
    "event_date",
    F.to_date("event_date")
)
print("Incoming Source Data:")
display(source_df)

Incoming Source Data:


id,event_date,amount
3,2024-01-03,300
1,2024-01-01,150


**STEP 3: FILTER LATE ARRIVING DATA**

In [0]:
filtered_df = source_df.filter(
    F.col("event_date") >= F.lit("2024-01-01").cast("date")
)
print("Filtered Source Data:")
display(filtered_df)

Filtered Source Data:


id,event_date,amount
3,2024-01-03,300
1,2024-01-01,150


**STEP 4: REMOVE DUPLICATES IN SOURCE DATA**

In [0]:
deduplicated_df = filtered_df.dropDuplicates(["id"])
display(deduplicated_df)

id,event_date,amount
3,2024-01-03,300
1,2024-01-01,150


**STEP 5: APPLY DELTA LAKE MERGE (UPSERT)**

In [0]:
delta_table = DeltaTable.forName(
    spark,
    table_name
)
delta_table.alias("target").merge(
    deduplicated_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()
print("Delta MERGE Completed Successfully!")

Delta MERGE Completed Successfully!


**STEP 6: VALIDATE FINAL DATA**

In [0]:
final_df = spark.table(table_name)
print("Final Processed Data:")
display(
    final_df.orderBy("event_date")
)

Final Processed Data:


id,event_date,amount
1,2024-01-01,150
2,2024-01-02,200
3,2024-01-03,300


**STEP 7: CHECK DUPLICATE RECORDS**

In [0]:
duplicate_df = final_df.groupBy("id") \
    .count() \
    .filter(F.col("count") > 1)
print("Duplicate Records:")
display(duplicate_df)

Duplicate Records:


id,count


**STEP 8: FINAL RECORD COUNT**

In [0]:
print("Total Records:", final_df.count())

Total Records: 3
